# NB01: Schema Discovery and Evidence Channel Characterization

**Purpose**: Before building the mapping pipeline, discover exactly what data is available in BERDL for linking UniProt proteins to ModelSEED reactions.

**Key questions**:
1. What are the distinct `status` values in `reaction`? Which ones are mass-balanced?
2. What cross-reference types (`db` values) exist in `uniprot_identifier`?
3. What does `u_seaver__msd_biochemistry` contain that the standard DB lacks?
4. What format are KEGG, BioCyc, Reactome entries in `uniprot_identifier`?
5. What does `reaction.abbreviation` actually contain (KEGG R-numbers? EC patterns?)?
6. What are the schemas of `curatedgene`, `interproscan_pathways`, `seedannotation`?

**Requires**: BERDL JupyterHub (Spark session)

In [1]:
import os, sys
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")
print('Spark session ready. Auto-broadcast disabled.')

Spark session ready. Auto-broadcast disabled.


## 1. ModelSEED Biochemistry: Reaction Status Values

The `reaction.status` column flags mass-balanced reactions. Enumerate all values.

In [2]:
reaction_status = spark.sql("""
    SELECT status, COUNT(*) as n_reactions
    FROM kbase_msd_biochemistry.reaction
    GROUP BY status
    ORDER BY n_reactions DESC
""").toPandas()

print('=== Reaction Status Distribution ===')
print(reaction_status.to_string(index=False))
print(f'\nTotal reactions: {reaction_status.n_reactions.sum():,}')
reaction_status.to_csv(f'{DATA_DIR}/reaction_status_counts.csv', index=False)

=== Reaction Status Distribution ===
                                                                           status  n_reactions
                                                                               OK        34343
                                                                     CPDFORMERROR         7090
                                                                            CI:-1         2436
                                                                             CI:1         1632
                                                                             CI:2         1411
                                                                            CI:-2         1261
                                                                            CI:-4          346
                                                                           MI:O:1          258
                                                                             CI:4          230
             

In [3]:
reaction_schema = spark.sql("DESCRIBE EXTENDED kbase_msd_biochemistry.reaction").toPandas()
print('=== kbase_msd_biochemistry.reaction schema ===')
print(reaction_schema[['col_name', 'data_type']].to_string(index=False))

=== kbase_msd_biochemistry.reaction schema ===
                    col_name                                                                    data_type
                abbreviation                                                                       string
                      deltag                                                                        float
                   deltagerr                                                                        float
                          id                                                                       string
                is_transport                                                                      boolean
                        name                                                                       string
               reversibility                                                                       string
                      source                                                                       string

## 2. Reaction Abbreviation Patterns

Check what `abbreviation` actually contains — KEGG R-numbers, EC-like patterns, MetaCyc IDs?

In [4]:
abbrev_stats = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN abbreviation IS NOT NULL AND abbreviation != '' THEN 1 ELSE 0 END) as has_abbreviation,
        SUM(CASE WHEN abbreviation RLIKE '^R[0-9]{5}' THEN 1 ELSE 0 END) as kegg_r_number,
        SUM(CASE WHEN abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+' THEN 1 ELSE 0 END) as has_ec_pattern,
        SUM(CASE WHEN abbreviation LIKE '%-RXN%' THEN 1 ELSE 0 END) as metacyc_rxn_pattern
    FROM kbase_msd_biochemistry.reaction
""").toPandas()

print('=== Abbreviation Pattern Analysis ===')
for col in abbrev_stats.columns:
    print(f'  {col}: {abbrev_stats[col].iloc[0]:,}')

=== Abbreviation Pattern Analysis ===
  total: 56,012
  has_abbreviation: 44,904
  kegg_r_number: 9,339
  has_ec_pattern: 2,596
  metacyc_rxn_pattern: 7,312


In [5]:
abbrev_samples = spark.sql("""
    SELECT abbreviation, COUNT(*) as n
    FROM kbase_msd_biochemistry.reaction
    WHERE abbreviation IS NOT NULL AND abbreviation != ''
    GROUP BY abbreviation
    ORDER BY n DESC
    LIMIT 30
""").toPandas()

print('=== Top 30 Most Common Abbreviations ===')
print(abbrev_samples.to_string(index=False))

=== Top 30 Most Common Abbreviations ===
                         abbreviation  n
            aureobasidin A synthetase  8
  menth-8-en-2-ol:NAD+ oxidoreductase  8
                     O-antigen ligase  6
      Acyl-homoserine-lactone acylase  6
flavanone,NADPH:oxygen oxidoreductase  5
                               R08125  5
       trehalose dimycolate synthesis  4
       epi-isozizaene 5-monooxygenase  4
                (side-chain-cleaving)  4
                               R01806  3
                               R02727  3
                               R02376  3
                               R02577  3
                               R08823  3
                               R02985  3
                               R02365  3
                               R00952  3
                               R02595  3
                               R08946  3
                               R08617  3
                               R01104  3
                               R00932  3
                

In [6]:
print('=== Sample abbreviation values (diverse patterns) ===')
samples = spark.sql("""
    (
        SELECT id, name, abbreviation, status, 'kegg_r' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation RLIKE '^R[0-9]{5}'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'ec_like' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+'
            AND NOT abbreviation RLIKE '^R[0-9]{5}'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'metacyc' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation LIKE '%-RXN%'
        LIMIT 5
    )
    UNION ALL
    (
        SELECT id, name, abbreviation, status, 'other' as pattern
        FROM kbase_msd_biochemistry.reaction
        WHERE abbreviation IS NOT NULL
            AND abbreviation != ''
            AND NOT abbreviation RLIKE '^R[0-9]{5}'
            AND NOT abbreviation RLIKE '[0-9]+\\.[0-9]+\\.[0-9]+\\.[0-9]+'
            AND abbreviation NOT LIKE '%-RXN%'
        LIMIT 5
    )
""").toPandas()

print(samples.to_string(index=False))

=== Sample abbreviation values (diverse patterns) ===


                    id                                                                                 name                                                                         abbreviation             status pattern
seed.reaction:rxn48569                                           Ubiquinol:ferricytochrome-c oxidoreductase                                                                               R02161     MI:C:-20/H:-32  kegg_r
seed.reaction:rxn48570                                                       NADH:ubiquinone oxidoreductase                                                                               R02163     MI:C:-20/H:-32  kegg_r
seed.reaction:rxn48571                                                  Succinate:ubiquinone oxidoreductase                                                                               R02164       MI:C:20/H:32  kegg_r
seed.reaction:rxn48572                          S-adenosyl-L-methione:demethylmenaquinone methyltransferase             

## 3. User's Augmented Biochemistry Database

Compare `u_seaver__msd_biochemistry` (6 tables) vs `kbase_msd_biochemistry` (5 tables).

In [7]:
std_tables = spark.sql("SHOW TABLES IN kbase_msd_biochemistry").toPandas()
print('=== kbase_msd_biochemistry tables ===')
print(std_tables.to_string(index=False))

print()

try:
    user_tables = spark.sql("SHOW TABLES IN u_seaver__msd_biochemistry").toPandas()
    print('=== u_seaver__msd_biochemistry tables ===')
    print(user_tables.to_string(index=False))

    std_set = set(std_tables['tableName'])
    user_set = set(user_tables['tableName'])
    extra = user_set - std_set
    if extra:
        print(f'\nExtra tables in user DB: {extra}')
        for t in extra:
            print(f'\n--- Schema for {t} ---')
            schema = spark.sql(f"DESCRIBE EXTENDED u_seaver__msd_biochemistry.{t}").toPandas()
            print(schema[['col_name', 'data_type']].to_string(index=False))
            print(f'\n--- Sample rows from {t} ---')
            sample = spark.sql(f"SELECT * FROM u_seaver__msd_biochemistry.{t} LIMIT 5").toPandas()
            print(sample.to_string(index=False))
    else:
        print('\nNo extra tables found.')
except Exception as e:
    print(f'Could not access u_seaver__msd_biochemistry: {e}')

=== kbase_msd_biochemistry tables ===
             namespace           tableName  isTemporary
kbase_msd_biochemistry            molecule        False
kbase_msd_biochemistry            reaction        False
kbase_msd_biochemistry             reagent        False
kbase_msd_biochemistry reaction_similarity        False
kbase_msd_biochemistry           structure        False



=== u_seaver__msd_biochemistry tables ===
                 namespace           tableName  isTemporary
u_seaver__msd_biochemistry            molecule        False
u_seaver__msd_biochemistry            reaction        False
u_seaver__msd_biochemistry            reagents        False
u_seaver__msd_biochemistry             reagent        False
u_seaver__msd_biochemistry reaction_similarity        False
u_seaver__msd_biochemistry           structure        False

Extra tables in user DB: {'reagents'}

--- Schema for reagents ---


                    col_name                                                                        data_type
                 reaction_id                                                                           string
                 compound_id                                                                           string
           compartment_index                                                                              int
               stoichiometry                                                                            float
                                                                                                             
# Detailed Table Information                                                                                 
                        Name                                spark_catalog.u_seaver__msd_biochemistry.reagents
                        Type                                                                         EXTERNAL
          

           reaction_id            compound_id  compartment_index  stoichiometry
seed.reaction:rxn00001 seed.compound:cpd00001                  0           -1.0
seed.reaction:rxn00001 seed.compound:cpd00012                  0           -1.0
seed.reaction:rxn00001 seed.compound:cpd00009                  0            2.0
seed.reaction:rxn00001 seed.compound:cpd00067                  0            1.0
seed.reaction:rxn00002 seed.compound:cpd00001                  0           -1.0


In [8]:
try:
    user_rxn_schema = spark.sql("DESCRIBE EXTENDED u_seaver__msd_biochemistry.reaction").toPandas()
    std_rxn_schema = spark.sql("DESCRIBE EXTENDED kbase_msd_biochemistry.reaction").toPandas()

    user_cols = set(user_rxn_schema['col_name'])
    std_cols = set(std_rxn_schema['col_name'])
    extra_cols = user_cols - std_cols
    if extra_cols:
        print(f'Extra columns in user reaction table: {extra_cols}')
        for col in extra_cols:
            sample = spark.sql(f"""
                SELECT `{col}`, COUNT(*) as n
                FROM u_seaver__msd_biochemistry.reaction
                WHERE `{col}` IS NOT NULL
                GROUP BY `{col}`
                ORDER BY n DESC
                LIMIT 20
            """).toPandas()
            print(f'\nTop values for {col}:')
            print(sample.to_string(index=False))
    else:
        print('Same columns in user and standard reaction tables.')
except Exception as e:
    print(f'Could not compare reaction schemas: {e}')

Same columns in user and standard reaction tables.


## 4. UniProt Databases: Table Inventory and Cross-Reference Types

Enumerate all tables in both UniProt databases, then characterize `uniprot_identifier`.

In [9]:
for db_name in ['refdata_uniprot', 'kbase_uniprot_kb']:
    try:
        tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
        print(f'=== {db_name} ({len(tables)} tables) ===')
        print(tables.to_string(index=False))
        print()

        for _, row in tables.iterrows():
            tbl = row['tableName']
            try:
                schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
                cols = ', '.join(schema['col_name'].tolist())
                cnt = spark.sql(f"SELECT COUNT(*) as n FROM {db_name}.{tbl}").collect()[0]['n']
                print(f'  {tbl} ({cnt:,} rows): {cols}')
            except Exception as e:
                print(f'  {tbl}: ERROR - {e}')
        print()
    except Exception as e:
        print(f'{db_name}: ERROR - {e}\n')

=== refdata_uniprot (13 tables) ===
      namespace              tableName  isTemporary
refdata_uniprot                cluster        False
refdata_uniprot          clustermember        False
refdata_uniprot            comment_xml        False
refdata_uniprot                 entity        False
refdata_uniprot   entity_x_data_source        False
refdata_uniprot   entity_x_publication        False
refdata_uniprot   entity_x_source_file        False
refdata_uniprot           evidence_xml        False
refdata_uniprot             identifier        False
refdata_uniprot identifier_partitioned        False
refdata_uniprot                   name        False
refdata_uniprot                protein        False
refdata_uniprot          reference_xml        False



  cluster (120,105,650 rows): cluster_id, name, cluster_type, protocol, _dlt_load_id, _dlt_id


  clustermember (259,569,754 rows): entity_id, cluster_id, is_representative, is_seed, _dlt_load_id, _dlt_id


  comment_xml (3,012,362 rows): entity_id, content, _dlt_load_id, _dlt_id


  entity (335,236,592 rows): entity_id, entity_type, data_source, updated, version, uniprot_created, uniprot_modified, _dlt_load_id, _dlt_id, data_source_entity_id, data_source_updated


...

  entity_x_publication (72,673,491 rows): entity_id, publication_id, _dlt_load_id, _dlt_id


  entity_x_source_file (982,170,266 rows): entity_id, data_source, source_file, _dlt_load_id, _dlt_id


  identifier (4,756,472,364 rows): entity_id, db, xref, description, _dlt_load_id, _dlt_id, relationship


  name (814,096,017 rows): entity_id, name, description, _dlt_load_id, _dlt_id


  protein (238,635,358 rows): protein_id, evidence_for_existence, length, hash, sequence, _dlt_load_id, _dlt_id



In [10]:
cached = f'{DATA_DIR}/identifier_db_counts.csv'
if os.path.exists(cached):
    db_values = pd.read_csv(cached)
    print('=== identifier: distinct db values with counts (from cache) ===')
    print(db_values.to_string(index=False))
    print(f'\nTotal db types: {len(db_values)}')
else:
    print('=== identifier: distinct db values with counts ===')
    print('(aggregating 4.35B rows — may take several minutes)')

    db_values = spark.sql("""
        SELECT db, COUNT(*) as n_rows, COUNT(DISTINCT entity_id) as n_entities
        FROM refdata_uniprot.identifier
        GROUP BY db
        ORDER BY n_rows DESC
    """).toPandas()

    print(db_values.to_string(index=False))
    print(f'\nTotal db types: {len(db_values)}')
    db_values.to_csv(cached, index=False)

=== identifier: distinct db values with counts (from cache) ===
                 db    n_rows  n_entities
           InterPro 667008339   163420970
                 GO 465900520   121734428
            genbank 457650928   186064283
            PANTHER 282804148   138928795
               Pfam 247326463   154008940
            UniProt 216632317   203130941
             Gene3D 216085145   131601237
          NCBITaxon 215130942   203130941
             refseq 214278166    98813634
          Proteomes 202508259   178990782
             SUPFAM 177805460   124848408
        AlphaFoldDB 167921046   158598889
            PROSITE 131882149    78788841
                CDD  88735059    67385521
             FunFam  82133922    44149947
              SMART  65519988    44453676
            OrthoDB  60470787    57115179
            NCBIfam  60363972    41307835
                 EC  38563959    34906907
             PRINTS  35436610    28583305
            ensembl  31199097     9123643
            

In [11]:
print('=== Sample entries for evidence-relevant db types ===\n')

evidence_dbs = ['EC', 'KEGG', 'BioCyc', 'Reactome', 'BRENDA', 'UniPathway', 'SABIO-RK',
                'InterPro', 'GO', 'Pfam', 'RHEA', 'MetaCyc', 'eggNOG']

for db_type in evidence_dbs:
    try:
        sample = spark.sql(f"""
            SELECT entity_id, db, xref, description
            FROM refdata_uniprot.identifier
            WHERE db = '{db_type}'
            LIMIT 10
        """).toPandas()
        if len(sample) > 0:
            print(f'--- db = {db_type} ({len(sample)} samples) ---')
            print(sample.to_string(index=False))
            print()
        else:
            print(f'--- db = {db_type}: NO ROWS ---\n')
    except Exception as e:
        print(f'--- db = {db_type}: ERROR - {e} ---\n')

=== Sample entries for evidence-relevant db types ===



--- db = EC (10 samples) ---
         entity_id db     xref description
    uniprot:W2IER8 EC 3.1.3.16        None
uniprot:A0A6I3SSR4 EC 2.1.2.11        None
uniprot:A0A6I3SVM6 EC  2.8.4.3        None
uniprot:A0A6I3T2Q3 EC 4.1.1.81        None
uniprot:A0A076ZC96 EC  3.5.3.6        None
uniprot:A0A5U9Q9A7 EC  2.7.7.3        None
uniprot:A0A8I1M6U1 EC  7.1.1.1        None
uniprot:A0A8I1SHJ8 EC  6.1.1.2        None
uniprot:A0A8I1SK53 EC  2.4.2.8        None
uniprot:A0AAW3HU49 EC  5.6.2.2        None



--- db = KEGG (10 samples) ---
         entity_id   db              xref description
uniprot:A0A4P7PA13 KEGG   pvk:EPZ47_00465        None
    uniprot:B7EHY1 KEGG dosa:Os03g0650000        None
    uniprot:Q5VQ36 KEGG dosa:Os06g0272900        None
    uniprot:Q5VQ36 KEGG       osa:4340733        None
    uniprot:Q688V5 KEGG       osa:4339650        None
uniprot:A0A9E6NKK7 KEGG  pze:HU754_016670        None
uniprot:A0A9E6NKL1 KEGG  pze:HU754_016745        None
uniprot:A0A9E6NLT6 KEGG  pze:HU754_022035        None
uniprot:A0A9E6NLU9 KEGG  pze:HU754_022280        None
uniprot:A0A9E6NM41 KEGG  pze:HU754_023020        None



--- db = BioCyc (10 samples) ---
     entity_id     db                             xref description
uniprot:B2AIX5 BioCyc CTAI977880:RALTA_RS29660-MONOMER        None
uniprot:B2AIX5 BioCyc CTAI977880:RALTA_RS29900-MONOMER        None
uniprot:Q8CLY6 BioCyc    SONE211586:G1GMP-1397-MONOMER        None
uniprot:Q8CLY6 BioCyc    SONE211586:G1GMP-1874-MONOMER        None
uniprot:Q8CLY6 BioCyc    SONE211586:G1GMP-1987-MONOMER        None
uniprot:Q8CLY6 BioCyc    SONE211586:G1GMP-2025-MONOMER        None
uniprot:Q8CME3 BioCyc    SONE211586:G1GMP-4423-MONOMER        None
uniprot:Q8CME3 BioCyc    SONE211586:G1GMP-4584-MONOMER        None
uniprot:Q7UA20 BioCyc     MetaCyc:TX72_RS00400-MONOMER        None
uniprot:A3R4R3 BioCyc            MetaCyc:MONOMER-21486        None



--- db = Reactome (10 samples) ---
     entity_id       db          xref description
uniprot:Q08DU3 Reactome R-BTA-1257604        None
uniprot:Q08DU3 Reactome R-BTA-5689880        None
uniprot:Q08DU3 Reactome R-BTA-6811558        None
uniprot:Q08DU3 Reactome R-BTA-9014843        None
uniprot:A4IF76 Reactome R-BTA-6798695        None
uniprot:A6QLL5 Reactome  R-BTA-210991        None
uniprot:A6QLL5 Reactome R-BTA-5578775        None
uniprot:A6QLL5 Reactome  R-BTA-936837        None
uniprot:A6QQT5 Reactome  R-BTA-674695        None
uniprot:A6QQT5 Reactome R-BTA-6796648        None



...

--- db = GO (10 samples) ---
         entity_id db    xref description
uniprot:A0A5C8CIZ6 GO 0009055        None
uniprot:A0A5C8CIZ6 GO 0010181        None
uniprot:A0A5C8CIZ6 GO 0070819        None
uniprot:A0A5C8CIZ6 GO 0006783        None
uniprot:A0A5Z7YRL7 GO 0005886        None
uniprot:A0A5Z7YRL7 GO 0050660        None
uniprot:A0A6A3K7A5 GO 0005829        None
uniprot:A0A6A3K7A5 GO 0005634        None
uniprot:A0A6A3K7A5 GO 0048471        None
uniprot:A0A6A3K7A5 GO 0005096        None



--- db = Pfam (10 samples) ---
         entity_id   db    xref description
uniprot:A0A068QWV2 Pfam PF04185        None
uniprot:A0A068QWV2 Pfam PF05506        None
uniprot:A0A1I0A2X9 Pfam PF04346        None
uniprot:A0A1J3HKS4 Pfam PF00141        None
uniprot:A0A3P3WYY6 Pfam PF00574        None
uniprot:A0A514TR18 Pfam PF00602        None
uniprot:A0A6I1B2L6 Pfam PF17851        None
uniprot:A0A6I1B2L6 Pfam PF04616        None
uniprot:A0A6M2ZW58 Pfam PF00602        None
uniprot:A0A0B4MAJ1 Pfam PF00115        None



--- db = RHEA: NO ROWS ---



--- db = MetaCyc: NO ROWS ---



--- db = eggNOG (10 samples) ---
         entity_id     db        xref description
    uniprot:Q688V5 eggNOG ENOG502RXG8        None
    uniprot:M0LBK4 eggNOG  arCOG02722        None
    uniprot:M0LHM6 eggNOG  arCOG00980        None
    uniprot:M0LIK7 eggNOG  arCOG08899        None
    uniprot:M0LRI9 eggNOG  arCOG04249        None
    uniprot:M0LS37 eggNOG  arCOG01808        None
    uniprot:M0LUR3 eggNOG  arCOG00571        None
uniprot:A0A0Q2G4Q3 eggNOG     COG0559        None
    uniprot:J4Z192 eggNOG     COG0346        None
uniprot:A0A095KXS6 eggNOG     COG3515        None



## 5. UniRef Databases

Check what UniRef clustering data is available for protein family grouping.

In [12]:
uniref_dbs = spark.sql("SHOW DATABASES").toPandas()
uniref_dbs = uniref_dbs[uniref_dbs['namespace'].str.contains('uniref', case=False, na=False)]
print('=== UniRef databases ===')
print(uniref_dbs.to_string(index=False))

for _, row in uniref_dbs.iterrows():
    db_name = row['namespace']
    tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
    print(f'\n--- {db_name} ({len(tables)} tables) ---')
    for _, trow in tables.iterrows():
        tbl = trow['tableName']
        try:
            schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
            cols = ', '.join(schema['col_name'].tolist())
            print(f'  {tbl}: {cols}')
        except Exception as e:
            print(f'  {tbl}: ERROR - {e}')

=== UniRef databases ===
                namespace
refdata_uniref100_2025_03
 refdata_uniref50_2025_03
 refdata_uniref50_2026_01
 refdata_uniref90_2025_03
 refdata_uniref90_2026_01



--- refdata_uniref100_2025_03 (4 tables) ---


  cluster: cluster_id, name, cluster_type, protocol, _dlt_load_id, _dlt_id


  clustermember: entity_id, cluster_id, is_representative, is_seed, _dlt_load_id, _dlt_id


  entity: entity_id, entity_type, data_source, data_source_entity_id, data_source_updated, updated, _dlt_load_id, _dlt_id


...

  entity: entity_id, entity_type, data_source, data_source_entity_id, data_source_updated, updated, _dlt_load_id, _dlt_id


  entity_x_source_file: entity_id, data_source, source_file, _dlt_load_id, _dlt_id

--- refdata_uniref90_2026_01 (4 tables) ---


  cluster: cluster_id, name, cluster_type, description, protocol


  entity: entity_id, entity_type, data_source, data_source_entity_id, data_source_updated, updated
  entity_x_source_file: entity_id, data_source, source_file


  clustermember: cluster_id, is_representative, is_seed, entity_id


## 6. PaperBLAST: curatedgene Schema

Understand what curated gene→reaction data is available.

In [13]:
pb_tables = spark.sql("SHOW TABLES IN kescience_paperblast").toPandas()
print('=== kescience_paperblast tables ===')
print(pb_tables.to_string(index=False))

for tbl in ['curatedgene', 'seqtoduplicate', 'gene']:
    try:
        schema = spark.sql(f"DESCRIBE kescience_paperblast.{tbl}").toPandas()
        print(f'\n--- {tbl} schema ---')
        print(schema[['col_name', 'data_type']].to_string(index=False))

        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_paperblast.{tbl}").collect()[0]['n']
        print(f'Rows: {cnt:,}')

        print(f'\n--- {tbl} sample (5 rows) ---')
        sample = spark.sql(f"SELECT * FROM kescience_paperblast.{tbl} LIMIT 5").toPandas()
        print(sample.to_string(index=False))
    except Exception as e:
        print(f'{tbl}: ERROR - {e}')
    print()

=== kescience_paperblast tables ===
           namespace      tableName  isTemporary
kescience_paperblast      genepaper        False
kescience_paperblast           gene        False
kescience_paperblast        snippet        False
kescience_paperblast    paperaccess        False
kescience_paperblast    curatedgene        False
kescience_paperblast   curatedpaper        False
kescience_paperblast seqtoduplicate        False
kescience_paperblast           site        False
kescience_paperblast       hassites        False
kescience_paperblast     seqhassite        False
kescience_paperblast      pdbligand        False
kescience_paperblast   pdbclustinfo        False
kescience_paperblast        generif        False
kescience_paperblast           uniq        False



--- curatedgene schema ---
      col_name data_type
            db    string
        protId    string
           id2    string
          name    string
          desc    string
      organism    string
protein_length    string
       comment    string


Rows: 255,096

--- curatedgene sample (5 rows) ---


       db protId         id2 name                                                                                             desc                                                                    organism protein_length                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

Rows: 284,104

--- seqtoduplicate sample (5 rows) ---


 sequence_id      duplicate_id
 VIMSS123152 SwissProt::Q9F9B1
 VIMSS123152      TCDB::Q9F9B1
 NP_008970.2            Q16825
 NP_008970.2    BRENDA::Q16825
VIMSS1293392      VIMSS1290560


--- gene schema ---
      col_name data_type
        geneId    string
      organism    string
protein_length       int
          desc    string


Rows: 1,135,366

--- gene sample (5 rows) ---


geneId     organism  protein_length                                                                 desc
B1N7B8 Homo sapiens             107 Cryocrystalglobulin CC1 kappa light chain variable region (Fragment)
A6PVK7 Homo sapiens            1370                                              Shortage in chiasmata 1
B7Z6P1 Homo sapiens             343        cDNA FLJ53662, highly similar to Actin, alpha skeletal muscle
B7ZAY9 Homo sapiens             280                                                    Cysteine protease
C9JS59 Homo sapiens             721                                   RING-type E3 ubiquitin transferase



In [14]:
print('=== curatedgene: distinct db values ===')
try:
    cg_dbs = spark.sql("""
        SELECT db, COUNT(*) as n
        FROM kescience_paperblast.curatedgene
        GROUP BY db
        ORDER BY n DESC
    """).toPandas()
    print(cg_dbs.to_string(index=False))
except Exception as e:
    print(f'Error: {e}')

=== curatedgene: distinct db values ===


        db      n
 SwissProt 110171
    biolip  42571
    BRENDA  33012
   metacyc  12700
    REBASE  12388
       ENA   9251
      CAZy   8878
      TCDB   8509
CharProtDB   8021
    ecocyc   4198
regprecise   3159
    reanno   1885
  prodoric    353


## 7. InterPro: protein2ipr and interproscan_pathways

Check what domain→EC→reaction evidence is available.

In [15]:
ip_tables = spark.sql("SHOW TABLES IN kescience_interpro").toPandas()
print('=== kescience_interpro tables ===')
print(ip_tables.to_string(index=False))

for tbl in ip_tables['tableName'].tolist():
    try:
        schema = spark.sql(f"DESCRIBE kescience_interpro.{tbl}").toPandas()
        print(f'\n--- {tbl} schema ---')
        print(schema[['col_name', 'data_type']].to_string(index=False))

        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_interpro.{tbl}").collect()[0]['n']
        print(f'Rows: {cnt:,}')
    except Exception as e:
        print(f'{tbl}: ERROR - {e}')
    print()

=== kescience_interpro tables ===
         namespace   tableName  isTemporary
kescience_interpro protein2ipr        False
kescience_interpro       entry        False
kescience_interpro  go_mapping        False



--- protein2ipr schema ---
   col_name data_type
uniprot_acc    string
     ipr_id    string
   ipr_desc    string
 source_acc    string
      start       int
       stop       int


Rows: 1,175,529,272


--- entry schema ---
  col_name data_type
    ipr_id    string
entry_type    string
entry_name    string


Rows: 51,489


--- go_mapping schema ---
col_name data_type
  ipr_id    string
   go_id    string
 go_name    string


Rows: 30,200



In [16]:
print('=== InterPro entries with EC associations ===')
try:
    schema = spark.sql("DESCRIBE kescience_interpro.entry").toPandas()
    print('entry schema:', ', '.join(schema['col_name'].tolist()))
    print()

    sample = spark.sql("""
        SELECT *
        FROM kescience_interpro.entry
        LIMIT 10
    """).toPandas()
    print(sample.to_string(index=False))
except Exception as e:
    print(f'Error: {e}')

=== InterPro entries with EC associations ===
entry schema: ipr_id, entry_type, entry_name



   ipr_id  entry_type                                                         entry_name
IPR000126 Active_site                    Serine proteases, V8 family, serine active site
IPR000138 Active_site                       Hydroxymethylglutaryl-CoA lyase, active site
IPR000169 Active_site                           Cysteine peptidase, cysteine active site
IPR000180 Active_site                                  Membrane dipeptidase, active site
IPR000189 Active_site                          Prokaryotic transglycosylase, active site
IPR000590 Active_site             Hydroxymethylglutaryl-coenzyme A synthase, active site
IPR001252 Active_site                                  Malate dehydrogenase, active site
IPR001345 Active_site           Phosphoglycerate/bisphosphoglycerate mutase, active site
IPR001497 Active_site Methylated-DNA-[protein]-cysteine S-methyltransferase, active site
IPR001555 Active_site           Phosphoribosylglycinamide formyltransferase, active site


In [17]:
print('=== interproscan_pathways: check for KEGG/MetaCyc pathway annotations ===')

for db_name in ['kbase_ke_pangenome']:
    try:
        tables = spark.sql(f"SHOW TABLES IN {db_name}").toPandas()
        ips_tables = tables[tables['tableName'].str.contains('interpro', case=False)]
        if len(ips_tables) > 0:
            print(f'InterPro-related tables in {db_name}:')
            print(ips_tables.to_string(index=False))
            for _, row in ips_tables.iterrows():
                tbl = row['tableName']
                schema = spark.sql(f"DESCRIBE {db_name}.{tbl}").toPandas()
                print(f'\n  {tbl}: {list(schema["col_name"])}')
                sample = spark.sql(f"SELECT * FROM {db_name}.{tbl} LIMIT 5").toPandas()
                print(sample.to_string(index=False))
        else:
            print(f'No InterPro tables in {db_name}')
    except Exception as e:
        print(f'{db_name}: {e}')

=== interproscan_pathways: check for KEGG/MetaCyc pathway annotations ===
InterPro-related tables in kbase_ke_pangenome:
         namespace             tableName  isTemporary
kbase_ke_pangenome  interproscan_domains        False
kbase_ke_pangenome       interproscan_go        False
kbase_ke_pangenome interproscan_pathways        False



  interproscan_domains: ['gene_cluster_id', 'md5', 'seq_len', 'analysis', 'signature_acc', 'signature_desc', 'start', 'stop', 'score', 'ipr_acc', 'ipr_desc']


 gene_cluster_id                              md5  seq_len    analysis    signature_acc                         signature_desc  start  stop       score   ipr_acc                         ipr_desc
VBKP01000272.1_2 7771c258ac5cd5a6d53ce21756f519c1      217      Gene3D G3DSA:3.90.10.10                          Cytochrome C3    119   217     6.4E-11      None                             None
VBKP01000272.1_2 7771c258ac5cd5a6d53ce21756f519c1      217     PANTHER        PTHR39425               LIPOPROTEIN CYTOCHROME C     13   217     2.6E-27      None                             None
VBKP01000272.1_2 7771c258ac5cd5a6d53ce21756f519c1      217         CDD          cd08168                           Cytochrom_C3     50   170 3.75979E-14      None                             None
VBKP01000272.1_2 7771c258ac5cd5a6d53ce21756f519c1      217        Pfam          PF14522 Cytochrome c7 and related cytochrome c    127   217      0.0014 IPR029467               Cytochrome c7-like
VBKP01000272.1_2 7771c258


  interproscan_go: ['gene_cluster_id', 'go_id', 'go_source', 'n_supporting_analyses']


       gene_cluster_id      go_id go_source  n_supporting_analyses
NZ_NHRD01000001.1_1500 GO:0016491  InterPro                      1
NZ_NHRD01000001.1_1501 GO:0005524  InterPro                      3
NZ_NHRD01000001.1_1501 GO:0005886   PANTHER                      1
NZ_NHRD01000001.1_1501 GO:0016887  InterPro                      3
NZ_NHRD01000001.1_1501 GO:0022857   PANTHER                      1



  interproscan_pathways: ['gene_cluster_id', 'pathway_db', 'pathway_id', 'n_supporting_analyses']


 gene_cluster_id pathway_db pathway_id  n_supporting_analyses
BLME01000050.1_3    MetaCyc   PWY-7884                      2
BLME01000056.1_1    MetaCyc   PWY-7884                      5
BLME01000058.1_1    MetaCyc   PWY-6902                      2
BLME01000058.1_1    MetaCyc   PWY-7822                      2
BLME01000058.1_1    MetaCyc   PWY-7883                      2


## 8. Fitness Browser: SEED Annotations

Check `seedannotation` and `seedclass` for SEED role→reaction links.

In [18]:
for tbl in ['seedannotation', 'seedclass', 'besthitkegg', 'besthitmetacyc', 'besthitswissprot']:
    try:
        schema = spark.sql(f"DESCRIBE kescience_fitnessbrowser.{tbl}").toPandas()
        cnt = spark.sql(f"SELECT COUNT(*) as n FROM kescience_fitnessbrowser.{tbl}").collect()[0]['n']
        print(f'--- {tbl} ({cnt:,} rows) ---')
        print(f'Columns: {list(schema["col_name"])}')

        sample = spark.sql(f"SELECT * FROM kescience_fitnessbrowser.{tbl} LIMIT 3").toPandas()
        print(sample.to_string(index=False))
        print()
    except Exception as e:
        print(f'{tbl}: ERROR - {e}\n')

--- seedannotation (177,519 rows) ---
Columns: ['orgId', 'locusId', 'seed_desc']


  orgId       locusId                                          seed_desc
Pedo557 CA265_RS14400 DNA topoisomerase IB (poxvirus type) (EC 5.99.1.2)
Pedo557 CA265_RS14405      Methionyl-tRNA formyltransferase (EC 2.1.2.9)
Pedo557 CA265_RS14410           Aminodeoxychorismate lyase (EC 4.1.3.38)



--- seedclass (61,874 rows) ---
Columns: ['orgId', 'locusId', 'type', 'num']


          orgId   locusId type      num
acidovorax_3H11  Ac3H11_7    1  2.3.2.2
acidovorax_3H11 Ac3H11_10    1 4.2.1.70
acidovorax_3H11 Ac3H11_13    1 4.2.1.51



--- besthitkegg (200,074 rows) ---
Columns: ['orgId', 'locusId', 'keggOrg', 'keggId', 'identity']


orgId     locusId keggOrg      keggId identity
Ponti CA264_18865     shg  Sph21_2714     51.7
Ponti CA264_18875     bbe BBR47_27060     51.5
Ponti CA264_18880     cac    CA_C1202     71.7



--- besthitmetacyc (59,723 rows) ---
Columns: ['orgId', 'locusId', 'protId', 'identity', 'desc', 'rxnId', 'ecnum']


          orgId   locusId        protId identity                                                                       desc                                  rxnId     ecnum
acidovorax_3H11 Ac3H11_10 G6581-MONOMER     58.2 23S rRNA pseudouridine2457 synthase (Escherichia coli K-12 substr. MG1655)                              RXN-11834 5.4.99.20
acidovorax_3H11 Ac3H11_13 MONOMER-17135     39.3                        arogenate dehydratase (Pseudomonas aeruginosa PAO1)                 PREPHENATEDEHYDRAT-RXN  4.2.1.51
acidovorax_3H11 Ac3H11_13 MONOMER-17135     39.3                        arogenate dehydratase (Pseudomonas aeruginosa PAO1) CARBOXYCYCLOHEXADIENYL-DEHYDRATASE-RXN  4.2.1.91



--- besthitswissprot (79,180 rows) ---
Columns: ['orgId', 'locusId', 'sprotAccession', 'sprotId', 'identity']


          orgId     locusId sprotAccession    sprotId identity
acidovorax_3H11   Ac3H11_34         P13254 MEGL_PSEPU     67.2
acidovorax_3H11 Ac3H11_4772         B9MDZ9 RPPH_ACIET     89.1
acidovorax_3H11 Ac3H11_3802         P27273 FDHB_WOLSU     65.5



## 9. eggNOG and bakta: EC/KEGG/UniRef Annotations

In [19]:
print('=== eggnog_mapper_annotations: EC/KEGG coverage ===')
ec_coverage = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN EC IS NOT NULL AND EC != '' AND EC != '-' THEN 1 ELSE 0 END) as has_ec,
        SUM(CASE WHEN KEGG_Reaction IS NOT NULL AND KEGG_Reaction != '' AND KEGG_Reaction != '-' THEN 1 ELSE 0 END) as has_kegg_rxn,
        SUM(CASE WHEN KEGG_ko IS NOT NULL AND KEGG_ko != '' AND KEGG_ko != '-' THEN 1 ELSE 0 END) as has_kegg_ko,
        SUM(CASE WHEN BiGG_Reaction IS NOT NULL AND BiGG_Reaction != '' AND BiGG_Reaction != '-' THEN 1 ELSE 0 END) as has_bigg
    FROM kbase_ke_pangenome.eggnog_mapper_annotations
""").toPandas()

total = ec_coverage['total'].iloc[0]
for col in ec_coverage.columns:
    val = ec_coverage[col].iloc[0]
    pct = val / total * 100 if total > 0 else 0
    print(f'  {col}: {val:,} ({pct:.1f}%)')

=== eggnog_mapper_annotations: EC/KEGG coverage ===


  total: 93,558,330 (100.0%)
  has_ec: 25,962,995 (27.8%)
  has_kegg_rxn: 20,286,023 (21.7%)
  has_kegg_ko: 51,127,232 (54.6%)
  has_bigg: 2,121,872 (2.3%)


In [20]:
print('=== bakta_annotations: EC/UniRef coverage ===')
bakta_coverage = spark.sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN ec IS NOT NULL AND ec != '' THEN 1 ELSE 0 END) as has_ec
    FROM kbase_ke_pangenome.bakta_annotations
""").toPandas()

total = bakta_coverage['total'].iloc[0]
for col in bakta_coverage.columns:
    val = bakta_coverage[col].iloc[0]
    pct = val / total * 100 if total > 0 else 0
    print(f'  {col}: {val:,} ({pct:.1f}%)')

print('\n=== bakta_db_xrefs: distinct db values ===')
bakta_dbs = spark.sql("""
    SELECT db, COUNT(*) as n
    FROM kbase_ke_pangenome.bakta_db_xrefs
    GROUP BY db
    ORDER BY n DESC
""").toPandas()
print(bakta_dbs.to_string(index=False))

=== bakta_annotations: EC/UniRef coverage ===


  total: 132,538,155 (100.0%)
  has_ec: 19,040,536 (14.4%)

=== bakta_db_xrefs: distinct db values ===


         db         n
     UniRef 242260603
         SO 102373648
    UniParc  61464352
     RefSeq  50046460
         GO  45014616
        COG  20112326
       PFAM  18807208
       KEGG  16748620
         EC  15177756
 BlastRules    223876
    NCBIFam     42742
NCBIProtein     40266
       VFDB     39150
         IS     24854


## 10. Cache Small Reference Tables

Download the reaction, molecule, and reagent tables for local use in downstream notebooks.

In [21]:
reactions = spark.sql("SELECT * FROM kbase_msd_biochemistry.reaction").toPandas()
reactions.to_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', index=False)
print(f'Cached {len(reactions):,} reactions to data/reactions_all.tsv')

molecules = spark.sql("SELECT * FROM kbase_msd_biochemistry.molecule").toPandas()
molecules.to_csv(f'{DATA_DIR}/molecules_all.tsv', sep='\t', index=False)
print(f'Cached {len(molecules):,} molecules to data/molecules_all.tsv')

reagents = spark.sql("SELECT * FROM kbase_msd_biochemistry.reagent").toPandas()
reagents.to_csv(f'{DATA_DIR}/reagents_all.tsv', sep='\t', index=False)
print(f'Cached {len(reagents):,} reagents to data/reagents_all.tsv')

Cached 56,012 reactions to data/reactions_all.tsv


Cached 45,708 molecules to data/molecules_all.tsv


Cached 262,517 reagents to data/reagents_all.tsv


## 11. Summary: Evidence Channel Viability

Based on all discoveries above, summarize which channels are viable and what the next steps are.

In [22]:
print('='*60)
print('EVIDENCE CHANNEL VIABILITY SUMMARY')
print('='*60)
print()
print('Review outputs above and fill in this summary:')
print()
print('Channel 1 (KEGG via uniprot_identifier):  [check db values above]')
print('Channel 2 (BioCyc via uniprot_identifier): [check db values above]')
print('Channel 3 (EC numbers):                    [check all EC sources above]')
print('Channel 4 (PaperBLAST):                    [check curatedgene schema above]')
print('Channel 5 (InterPro):                      [check protein2ipr above]')
print('Channel 6 (SEED annotations):              [check seedannotation above]')
print('Channel 7 (Fuzzy name matching):            Always available (reaction.name)')
print()
print('Mass-balanced reactions:                    [check status values above]')
print('User DB extras:                            [check u_seaver diff above]')
print()
print('Next: Revise RESEARCH_PLAN.md based on these findings,')
print('      then build NB02-NB06 for confirmed channels.')

EVIDENCE CHANNEL VIABILITY SUMMARY

Review outputs above and fill in this summary:

Channel 1 (KEGG via uniprot_identifier):  [check db values above]
Channel 2 (BioCyc via uniprot_identifier): [check db values above]
Channel 3 (EC numbers):                    [check all EC sources above]
Channel 4 (PaperBLAST):                    [check curatedgene schema above]
Channel 5 (InterPro):                      [check protein2ipr above]
Channel 6 (SEED annotations):              [check seedannotation above]
Channel 7 (Fuzzy name matching):            Always available (reaction.name)

Mass-balanced reactions:                    [check status values above]
User DB extras:                            [check u_seaver diff above]

Next: Revise RESEARCH_PLAN.md based on these findings,
      then build NB02-NB06 for confirmed channels.
